## Test de trajectoires des navires de pêche présents dans le parc 

Hypothèse testée: Des bateaux de pêche éteignent leur AIS dans le parc?

Type de bateaux: pêche palangrière

In [1]:
import os
import numpy as np

import pandas as pd
import geopandas as gpd

import datetime
import time

import matplotlib.pyplot as plt
import seaborn as sns
import folium

import re
from html import escape
import colorsys

Cannot find header.dxf (GDAL_DATA is not defined)


In [2]:
# load the data path
os.chdir('C:/Users/anael/Documents/PNMCCA_Marbec_2026/traj_bateaux_PNMCCA/')

## Trajectoires GFW
### bateaux identifié sur GFW directement
Bateaux de pêche (palangriers) qui font au moins un passage à un moment t dans le parc marin (pêche ou non).

- Italia II > 2024
- Madonna del Carmine > 2022-2026
- Madonna di Fatima > 2023
- Ammiraglia > 2023
- ct2895 > 2022

### Bateaux ajouté à partir des données AIS 2025 fournit par le parc
2 bateaux ajoutés (palangriers). 

- Pablo I > 2024-2026
- Feli > 2025-2026

Madonna del Carmine bien pésent dans leur données aussi. 6 bateaux de pêches présents dans les données AIS du parc mais pas inclus ici car pas palangriers (chalutier ou senner), traverse juste une ou deux fois le parc, ou mauvaise classif (bateaux de loisir, fishery research vessel).

In [3]:
# navires trouvé à partir de GFW uniquement 
ammiraglia = pd.read_csv('./raw_traj_GFW/ammiraglia-2023-2024.csv')
ct2895 = pd.read_csv('./raw_traj_GFW/ct2895-2022.csv')
italiaII = pd.read_csv('./raw_traj_GFW/italiaII-2023.csv')
madonnadelcarmine = pd.read_csv('./raw_traj_GFW/madonnadelcarmine-2022-2026.csv')
madonnadifatima = pd.read_csv('./raw_traj_GFW/madonnadifatima-2023.csv')

# 2 navires ajourter suite grâce aux détections AIS fournit par le parc + traj de GFW
feli = pd.read_csv('./raw_traj_GFW/feli-2025-2026.csv')
pabloI = pd.read_csv('./raw_traj_GFW/pabloI-2024-2026.csv')

In [4]:
ais_data_list = [ammiraglia, ct2895, italiaII, madonnadelcarmine, madonnadifatima, feli, pabloI]

## Tester sur ces bateaux l'hypothèse de coupure d'AIS 
Extraire les points AIS espacé de 24h ou plus (= les potentielles coupures d'AIS).

In [5]:
ais_data_list[0]

,lon,lat,course,timestamp,speed,depth,seg_id
0,15.800467,38.315630,319.799988,2023-01-03T03:55:48.000Z,7.8,-414,247133620-2023-01-03T03:55:48.000000Z-1
1,15.747247,38.355190,308.799988,2023-01-03T04:19:36.000Z,8.7,-358,247133620-2023-01-03T03:55:48.000000Z-1
2,15.719147,38.382538,320.000000,2023-01-03T04:34:37.000Z,8.3,-438,247133620-2023-01-03T03:55:48.000000Z-1
3,15.687755,38.407688,316.299988,2023-01-03T04:49:47.000Z,8.2,-459,247133620-2023-01-03T03:55:48.000000Z-1
4,15.620213,38.445623,272.700012,2023-01-03T05:37:56.000Z,1.6,-627,247133620-2023-01-03T03:55:48.000000Z-1
...,...,...,...,...,...,...,...
7983,15.722483,38.391087,149.699997,2023-12-29T15:00:41.000Z,6.1,-440,247133620-2023-12-27T03:55:41.000000Z-1
7984,15.740368,38.370843,142.300003,2023-12-29T15:14:32.000Z,8.0,-363,247133620-2023-12-27T03:55:41.000000Z-1
7985,15.791050,38.319625,141.000000,2023-12-29T15:41:02.000Z,9.0,-436,247133620-2023-12-27T03:55:41.000000Z-1
7986,15.803427,38.308580,135.000000,2023-12-29T15:47:11.000Z,8.7,-362,247133620-2023-12-27T03:55:41.000000Z-1


In [6]:
for i in np.arange(7):
    df = ais_data_list[i]
    df.loc[: ,'timestamp'] = pd.to_datetime(df.timestamp)

In [8]:
# fonction qui calcule l'intervalle de temps entre deux timestamp
def Tdiff(data_ais):    
    #create new columns shifting the timestamp_formated up and down by one
    data_ais['timestamp_down'] = data_ais['timestamp'].shift(1, axis=0)
    data_ais['timestamp_up'] = data_ais['timestamp'].shift(-1, axis=0)
    
    # calculate the mean time based on the new columns
    data_ais['T'] = (data_ais['timestamp_up'] - data_ais['timestamp_down'])/2
    
    #drop the column created to keep the df clean
    data_ais = data_ais.drop(['timestamp_down', 'timestamp_up'], axis=1)

    return data_ais

In [9]:
for i in np.arange(7):
    df = ais_data_list[i]
    df = Tdiff(df)
    df['T'] = pd.to_timedelta(df['T'])
    df['T_hour'] = df['T'].dt.total_seconds() / 3600 #transform T in a number of hour
    ais_data_list[i] = df

In [12]:
# selectionné les lignes où l'écart des timestamp est supérieur à 24h.
ais_ecarts_list = []

for i in range(7):
    values_list = []
    df = ais_data_list[i].copy()
    mask = df['T_hour'] > 24 
    indices = df.index[mask]# Indices des lignes qui remplissent la condition

    for idx in indices:     
        pos = df.index.get_loc(idx)# Position de la ligne dans le DataFrame
        if pos > 0:
            T_hour = df.iloc[pos]['T_hour']
            timestamp_A = df.iloc[pos-1]['timestamp']
            timestamp_B = df.iloc[pos]['timestamp']
            
            lon_A = df.iloc[pos]['lon']
            lat_A = df.iloc[pos]['lat']
            course_A = df.iloc[pos]['course']
            speed_A = df.iloc[pos]['speed']
            depth_A = df.iloc[pos]['depth']

            lon_B = df.iloc[pos-1]['lon']
            lat_B = df.iloc[pos-1]['lat']
            course_B = df.iloc[pos-1]['course']
            speed_B = df.iloc[pos-1]['speed']
            depth_B = df.iloc[pos-1]['depth']

            values_list.append((T_hour, timestamp_A, timestamp_B, 
                                lon_A, lat_A, course_A, speed_A, depth_A,
                               lon_B, lat_B, course_B, speed_B, depth_B,))
    ais_ecarts= pd.DataFrame(values_list, columns=['T_hour', 'timestamp_A', 'timestamp_B', 
                                                   'lon_A', 'lat_A', 'course_A', 'speed_A', 'depth_A',
                                                  'lon_B', 'lat_B', 'course_B', 'speed_B', 'depth_B']) # df 
    ais_ecarts_list.append(ais_ecarts)

In [24]:
# Mapper les deux points AIS en question (permet de voir où se produire ces potentielles coupures d'AIS: au port ou en mer...)
map_list = []
CARTO_API_KEY = "cb1_4uyn_1_dfa047c0e6c9462ab0eba889"

for num_bateau in range(7): 
    df = ais_ecarts_list[num_bateau].copy()
    
    # Vérifier les coordonnées disponibles
    all_lats = pd.concat([df["lat_A"], df["lat_B"]]).dropna()
    all_lons = pd.concat([df["lon_A"], df["lon_B"]]).dropna()
    
    m = folium.Map(
        location=[all_lats.mean(), all_lons.mean()],
        zoom_start=12,
        tiles=None
    )
    
    folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/"
        "rastertiles/light_all/{z}/{x}/{y}.png"
        f"?key={CARTO_API_KEY}"
    ),
    attr='© OpenStreetMap contributors © CARTO',
    name="CARTO Positron",
    max_zoom=20
    ).add_to(m)
    
    # Générer une couleur distincte pour chaque paire
    def get_color(i):
        hue = (i * 0.61803398875) % 1
        r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.85)
        return "#{:02x}{:02x}{:02x}".format(
            int(r * 255), int(g * 255), int(b * 255)
        )
    
    # Colonnes propres à chaque point
    cols_A = [c for c in df.columns if c.endswith("_A")]
    cols_B = [c for c in df.columns if c.endswith("_B")]
    
    def create_popup(row, cols, suffix, color):
        # T_hour est ajouté même s'il ne possède pas de suffixe
        html = f"""
        <div style="font-family: Arial; min-width: 220px;">
            <h4 style="color: {color}; margin-bottom: 8px;">
                Paire {row.name} — Point {suffix}
            </h4>
            <table style="border-collapse: collapse; width: 100%;">
        """
    
        # Afficher T_hour en premier
        fields = ["T_hour"] + [c for c in cols if c != "T_hour"]
    
        for col in fields:
            if col not in row.index:
                continue
    
            label = col[:-2] if col.endswith(("_A", "_B")) else col
            value = escape(str(row[col]))
    
            html += f"""
            <tr>
                <td style="padding: 4px; border-bottom: 1px solid #ddd;">
                    <b>{escape(label)}</b>
                </td>
                <td style="padding: 4px; border-bottom: 1px solid #ddd;">
                    {value}
                </td>
            </tr>
            """
    
        html += "</table></div>"
        return folium.Popup(html, max_width=400)
    
    # Ajouter les marqueurs avec des formes différentes pour A et B
    for i, (idx, row) in enumerate(df.iterrows()):
        color = get_color(i)
    
        # Point A : cercle
        if pd.notna(row["lat_A"]) and pd.notna(row["lon_A"]):
            folium.CircleMarker(
                location=[row["lat_A"], row["lon_A"]],
                radius=7,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.85,
                weight=2,
                tooltip=f"Paire {idx} — Point A | T_hour: {row['T_hour']}",
                popup=create_popup(row, cols_A, "A", color)
            ).add_to(m)
    
        # Point B : triangle
        if pd.notna(row["lat_B"]) and pd.notna(row["lon_B"]):
            folium.RegularPolygonMarker(
                location=[row["lat_B"], row["lon_B"]],
                number_of_sides=3,
                radius=9,
                rotation=0,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.85,
                tooltip=f"Paire {idx} — Point B | T_hour: {row['T_hour']}",
                popup=create_popup(row, cols_B, "B", color)
            ).add_to(m)
        # Relier les points A et B de chaque paire
        if (
            pd.notna(row["lat_A"]) and pd.notna(row["lon_A"])
            and pd.notna(row["lat_B"]) and pd.notna(row["lon_B"])
        ):
            folium.PolyLine(
                locations=[
                    [row["lat_A"], row["lon_A"]],
                    [row["lat_B"], row["lon_B"]]
                ],
                color=color,
                weight=2,
                opacity=0.7,
                tooltip=f"Paire {idx} — T_hour: {row['T_hour']}"
            ).add_to(m)
    # Adapter le cadrage à tous les points valides
    points = []
    for _, row in df.iterrows():
        for lat_col, lon_col in [
            ("lat_A", "lon_A"),
            ("lat_B", "lon_B")
        ]:
            if pd.notna(row[lat_col]) and pd.notna(row[lon_col]):
                points.append([row[lat_col], row[lon_col]])
    
    if points:
        m.fit_bounds(points)

    map_list.append(m)


# Resultats

Visualisation des cartes : 1 carte par navire, représentant les paires de points AIS successifs espacés de >24h (les potentielles coupures d'AIS).

🔵 Cercle = Point A : localisation du premier point de la paire.

🔷 Triangle = Point B : localisation du second point de la paire.

Ces deux points sont reliés par une ligne.
Chaque paire de points AIS ainsi définie est représentée d'une couleur différente.

<u>Informations interactives<u>

Survol d'un marqueur : affiche l'identifiant de la paire, le type de point (A ou B) et la valeur de T_hour qui est la durée entre les deux points AIS successifs.

Clic sur un marqueur : ouvre une fenêtre contenant les informations associées au point: date, coordonnées, cap, vitesse.

Survol d'une ligne : affiche l'identifiant de la paire et la valeur de T_hour.


In [16]:
list_names = ["ammiraglia", "ct2895", "italiaII", "madonnadelcarmine", "madonnadifatima", "feli", "pabloI"]

In [27]:
print(list_names[0])
display(map_list[0])

ammiraglia


In [26]:
print(list_names[1])
display(map_list[1])

ct2895


In [28]:
print(list_names[2])
display(map_list[2])

italiaII


In [29]:
print(list_names[3])
display(map_list[3])

madonnadelcarmine


In [30]:
print(list_names[4])
display(map_list[4])

madonnadifatima


In [31]:
print(list_names[5])
display(map_list[5])

feli


In [32]:
print(list_names[6])
display(map_list[6])

pabloI


In [33]:
dossier = 'C:\\Users\\anael\\Documents\\PNMCCA_Marbec_2026'

map_list[0].save(os.path.join(dossier, "map_1.html"))
map_list[1].save(os.path.join(dossier, "map_2.html"))
map_list[2].save(os.path.join(dossier, "map_3.html"))
map_list[3].save(os.path.join(dossier, "map_4.html"))
map_list[4].save(os.path.join(dossier, "map_5.html"))
map_list[5].save(os.path.join(dossier, "map_6.html"))
map_list[6].save(os.path.join(dossier, "map_7.html"))